<a href="https://colab.research.google.com/github/SFcrypt/Segsmaker-x/blob/main/notebook/maker/Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Iniciar Proyecto 🧽**

</details>
<img src="https://i.pinimg.com/originals/c1/bc/3d/c1bc3d6ba8b7a7c9ff037660e2e4f2c2.gif" width="150%" height="200‰" style="margin-bottom: 0;">

Automatiza la creación y configuración de carpetas en Google Drive, estableciendo una estructura organizada y personalizado para los proyectos de personajes.

<small>
  <a href="https://civitai.com" target="_blank" style="text-decoration:none;">
    <img src="https://img.shields.io/badge/9.0-353535?style=for-the-badge&logo=Github&label=Versi%C3%B3n&labelColor=292929" alt="My Civitai" width="110">
  </a>
</small>
<br>
<small>
  <a href="https://civitai.com" target="_blank" style="text-decoration:none;">
    <img src="https://img.shields.io/badge/7.0-353535?style=for-the-badge&logo=Pointy&logoColor=%23fafafa&label=Manager&labelColor=292929" alt="My Civitai" width="112">
  </a>
</small>

---

<details>
  <summary><font color=gray>Disclaimer</font></summary>

```diff
⭕ Descargo de responsabilidad
+ Herramientas para gestión de archivos y proyectos en Google Drive  
+ Cumplimiento de las directrices de Google Colab  
+ Adherencia a los Términos de servicio de Google Colab  
```

</details>

<details>
  <summary><font color=gray>Últimos cambios</font></summary>

```diff
+ Añadido soporte para etiquetado automático con WaifuDiffusion Tagger  
+ Mejorada la interfaz interactiva para filtrar y recortar imágenes  
+ Implementada función para renombrar y convertir imágenes en lote  
+ Integrada herramienta para comprimir y descomprimir archivos .zip  
+ Añadido conteo de archivos en carpetas y subcarpetas  
+ Simplificada la estructura de carpetas generadas en Google Drive  
+ Corregidos errores al montar Google Drive  
```

</details>

<details>
  <summary><font color=gray>Cambios antiguos</font></summary>

```diff
+ Optimizaciones en la compresión y descompresión de archivos  
+ Mejora en la configuración inicial de Colab  
+ Reducción del tiempo de instalación de dependencias  
+ Actualizaciones de bibliotecas críticas como `onnxruntime`, `ultralytics`, `Pillow`, y más  
+ Añadido soporte para conteo de archivos específicos (imágenes, textos, modelos)  
+ Simplificada la integración con Google Drive  
+ Mejor manejo de errores al crear carpetas en Google Drive  
```

In [ ]:
# configurar proyecto

#@markdown ### **Configuración del proyecto**
#@markdown este bloque inicializa el proyecto en google colab, prepara el entorno
#@markdown de trabajo y define la estructura principal donde se guardarán los archivos.

# módulos necesarios
import os, re, io, time
from IPython import get_ipython
import ipywidgets as widgets
from IPython.display import clear_output, display, HTML

#@markdown también descarga archivos desde **github**
#@markdown y muestra mensajes visuales para confirmar que cada paso se completó correctamente. <p>

# conectar google drive
if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# clonar colabui
if not os.path.exists("ColabUI"):
    !git clone https://github.com/SFcrypt/ColabUI.git
    clear_output()

# importar widgets visuales (global)
from ColabUI.Animated_box import (
    load_style, pink_button_download)

from ColabUI.Animated_progress import (
    Animated_progress)

# función configurar Nombre
def configurar_proyecto(b):
    clear_output(wait=True)

    global ruta_name
    nombre = nombre_input.value.strip().lower()

    # si no hay nombre salir
    if not nombre:
        return

    # definir rutas
    ruta_name  = nombre
    ruta_drive = "drive/MyDrive/"
    ruta_proy  = os.path.join(ruta_drive, "Loras")
    ruta_data  = os.path.join(ruta_proy, ruta_name, "dataset")

    os.makedirs(ruta_data, exist_ok=True)
    clear_output()
    load_style()

    # mostrar progreso con animación
    progress = Animated_progress(total=5,
    title="Creando proyecto")

    # pequeña animación
    for i in range(1, 8):
        time.sleep(0.3)
        progress.update(step=1, text=f"")

    # completar con título final
    progress.button_text = ruta_name
    progress.complete("Proyecto listo")

# Crear input centrado y botón rosa
init_btn, nombre_input = pink_button_download(
    title="Crear proyecto",
    btn_text="Crear",
    input_placeholder="Nombre del proyecto")

# vincular evento click
init_btn.on_click(configurar_proyecto)

# fin

In [ ]:
# Descargar Proyecto

#@markdown ### **Descargar proyecto**
#@markdown este script permite descargar un archivo .zip desde un link directo de google drive,
#@markdown descomprimirlo automáticamente en tu drive y eliminar el zip al finalizar.
#@markdown además, actualiza automáticamente el nombre del `proyecto`
#@markdown basado en el archivo zip, incluye barra de progreso personalizada.

# importaciones y autenticar Drive
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, zipfile, logging, os, re
from IPython.display import clear_output

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Estilo y ruta
load_style()
ruta_drive = "drive/MyDrive/"

# Widget descarga
download_btn, link_input = pink_button_download(
    title="Descargar proyecto",
    btn_text="Descargar",
    input_placeholder="Link del proyecto")

# Extraer ID del link
def extract_file_id_from_url(url):
    match = re.search(r"/file/d/([a-zA-Z0-9_-]+)", url)
    if not match: raise Exception("❌ Enlace incorrecto")
    return match.group(1)

# Descargar y extraer zip
def download_and_extract(file_id, nombre, ruta_destino):
    global ruta_name

    # Obtener info del archivo
    meta = drive_service.files().get(fileId=file_id, fields="size, name").execute()
    file_size = int(meta['size'])
    file_name = meta.get('name', nombre)
    output_path = os.path.join(ruta_destino, file_name)

    # Definir ruta_name como nombre del zip
    ruta_name = os.path.splitext(file_name)[0]
    progress = Animated_progress(total=file_size, title="Descargando")

    # Descargar archivo
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.FileIO(output_path, 'wb')
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        progress.update(int(status.progress() * file_size))

    # Extraer zip
    with zipfile.ZipFile(output_path, 'r') as zipf:
        for f in zipf.infolist():
            zipf.extract(f, ruta_destino)

    # Actualizar barra y eliminar zip
    progress.button_text = ruta_name
    progress.complete("Proyecto listo")
    os.remove(output_path)

# Descargar proyecto al click
def descargar_proyecto(b):
    clear_output(wait=True)
    Link = link_input.value.strip()
    if not Link: return
    try:
        file_id = extract_file_id_from_url(Link)
        download_and_extract(file_id, "", ruta_drive)
    except Exception as e:
        print(f"Error: {e}")

download_btn.on_click(descargar_proyecto)

# Fin

In [ ]:
# Comprimir Proyecto

#@markdown ### **Comprimir proyecto**
#@markdown este script permite comprimir automáticamente `dataset` de un proyecto en un archivo `zip`
#@markdown la compresión solo se realizará si la carpeta contiene más de 5 imágenes.
#@markdown es útil para hacer respaldos rápidos de tus proyectos y mantenerlos organizados.

# importaciones
import os, zipfile
from IPython.display import clear_output

# rutas dinámicas
ruta_drive  = "drive/MyDrive/"
ruta_proy   = os.path.join(ruta_drive, "Loras")
ruta_back   = os.path.join(ruta_drive, "Backup")
ruta_data   = os.path.join(ruta_proy, ruta_name, "dataset")

os.makedirs(ruta_data, exist_ok=True)
os.makedirs(ruta_back, exist_ok=True)

# función para comprimir 'dataset' solo si hay más de 5 imágenes
def comprimir_dataset(ruta_carpeta, ruta_zip):
    try:
        imagenes = [f for f in os.listdir(ruta_carpeta) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
        if len(imagenes) <= 5:
            print("error: ❌ imágenes insuficientes")
            return

        # obtener todos los archivos de dataset
        archivos = [os.path.join(r, f) for r, d, fs in os.walk(ruta_carpeta) for f in fs]

        # barra de progreso
        progress = Animated_progress(
        total=len(archivos),
        title="comprimiendo")

        # crear zip
        with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for f in archivos:
                arcname = os.path.join("Loras", ruta_name, "dataset", os.path.relpath(f, start=ruta_carpeta))
                zipf.write(f, arcname)
                progress.update(1)

        # actualizar botón con número de imágenes
        progress.button_text = f"{len(imagenes)} imágenes"
        progress.complete("Proyecto listo")

    except Exception as e:
        print(f"error: ❌ comprimiendo: {e}")

# ejecutar compresión automáticamente
ruta_zip = os.path.join(ruta_back, f"{ruta_name}.zip")
comprimir_dataset(ruta_data, ruta_zip)

#Fin


## **Crear Dataset 💣**
[![My Github](https://img.shields.io/badge/GitHub-292929?style=for-the-badge&logo=GitHub)](https://github.com/TuUsuario)
[![My Civitai](https://img.shields.io/badge/Colab-%23292929?style=for-the-badge&logo=googlecolab)](https://civitai.com)

Herramientas integrales para etiquetar, filtrar, recortar, renombrar, comprimir, descomprimir, y descargar. Optimiza la organización y el procesamiento de tus proyectos con IA y automatización.

<small>
  <a href="https://civitai.com" target="_blank" style="text-decoration:none;">
    <img src="https://img.shields.io/badge/8.0-353535?style=for-the-badge&logo=Github&label=Versi%C3%B3n&labelColor=292929" alt="My Civitai" width="110">
  </a>
</small>
<br>
<small>
  <a href="https://civitai.com" target="_blank" style="text-decoration:none;">
    <img src="https://img.shields.io/badge/5.5-353535?style=for-the-badge&logo=Pointy&logoColor=%23fafafa&label=Manager&labelColor=292929" alt="My Civitai" width="112">
  </a>
</small>

---

<details>
  <summary><font color=gray>Disclaimer</font></summary>

```diff
⭕ Descargo de responsabilidad
+ Herramientas para gestión de archivos y proyectos en Google Drive  
+ Cumplimiento de las directrices de Google Colab  
+ Adherencia a los Términos de servicio de Google Colab  
```

</details>

<details>
  <summary><font color=gray>Últimos cambios</font></summary>

```diff
+ Añadido soporte para etiquetado automático con WaifuDiffusion Tagger  
+ Mejorada la interfaz interactiva para filtrar y recortar imágenes  
+ Implementada función para renombrar y convertir imágenes en lote  
+ Integrada herramienta para comprimir y descomprimir archivos .zip  
+ Añadido conteo de archivos en carpetas y subcarpetas  
+ Simplificada la estructura de carpetas generadas en Google Drive  
+ Corregidos errores al montar Google Drive  
```

</details>

<details>
  <summary><font color=gray>Cambios antiguos</font></summary>

```diff
+ Optimizaciones en la compresión y descompresión de archivos  
+ Mejora en la configuración inicial de Colab  
+ Reducción del tiempo de instalación de dependencias  
+ Actualizaciones de bibliotecas críticas como `onnxruntime`, `ultralytics`, `Pillow`, y más  
+ Añadido soporte para conteo de archivos específicos (imágenes, textos, modelos)  
+ Simplificada la integración con Google Drive  
+ Mejor manejo de errores al crear carpetas en Google Drive  
```

### **Estraer capturas 🎥**

<img src="https://i.pinimg.com/originals/65/87/c0/6587c02f1ff0b4b8686ffc179fcacc1c.gif" width="150%" height="230‰" style="margin-bottom: 0;">

Combina dos funcionalidades principales descargar videos desde Google Drive y extraer frames de videos mp4 utilizando FFmpeg, Ofrece una interfaz interactiva para seleccionar archivos.

<small>
  <a style="text-decoration:none;">
    <img src="https://img.shields.io/badge/4.0-353535?style=for-the-badge&logo=Android&logoColor=%23fafafa&label=Versión&labelColor=292929" alt="Versión 1.0" width="100">
  </a>
</small>
<br>
<small>
  <a style="text-decoration:none;">
    <img src="https://img.shields.io/badge/2-353535?style=for-the-badge&logo=Github&label=Pasos&labelColor=292929" alt="Google Drive Management" width="75">
  </a>
</small>

---

<details>
  <summary><font color=gray>Últimos cambios</font></summary>

```diff
+ Interfaz interactiva para seleccionar archivos y configurar opciones.  
+ Barras de progreso personalizadas para descargas y extracción de frames.  
+ Soporte para aceleración por GPU en la extracción de frames.  
+ Validación de permisos y existencia de archivos.  
```

</details>  

<details>
  <summary><font color=gray>Cambios antiguos</font></summary>

```diff
+ Conexión automatizada a Google Drive desde Google Colab.  
+ Creación de rutas dinámicas para descargas y almacenamiento de frames.  
+ Integración de FFmpeg para la extracción de frames.  
+ Soporte para videos en formato mp4 y gestión de carpetas en Google Drive.  
```

---

In [ ]:
# Descargar Videos de Drive

#@markdown ### **Descargar video de drive**
#@markdown este script permite descargar videos desde una carpeta compartida de google drive.
#@markdown el usuario puede pegar el link del video y descargarlo en una carpeta local o en google drive.
#@markdown asegúrate de tener los permisos necesarios para acceder a la carpeta y descargar los archivos.

# importaciones
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import os, io, re, shutil, logging
from IPython.display import clear_output

# autenticación para google drive
auth.authenticate_user()
drive_service = build('drive', 'v3')
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

# crear widget de descarga (botón + input)
download_btn, link_input = pink_button_download(
    title="Descargar video",
    btn_text="Descargar",
    input_placeholder="link del video")

# carpeta donde se guardarán los videos
ruta_video = "/content/FFmpeg"
os.makedirs(ruta_video, exist_ok=True)

# función para extraer el id del archivo desde el link
def extract_file_id_from_url(url):
    match = re.search(r"/file/d/([a-zA-Z0-9_-]+)", url)
    if not match:
        raise Exception("no se pudo extraer el id del archivo")
    return match.group(1)

# función para descargar archivo desde drive con barra de progreso
def download_from_drive(file_id):
    try:# borrar carpeta antes de iniciar nueva descarga
        if os.path.exists(ruta_video):
            shutil.rmtree(ruta_video)
        os.makedirs(ruta_video, exist_ok=True)

        # obtener metadata del archivo
        meta = drive_service.files().get(fileId=file_id, fields="size, name").execute()
        file_size = int(meta['size'])
        file_name = meta['name']

        clear_output()
        progress = Animated_progress(total=file_size, title="Descargando")

        # descargar archivo
        request = drive_service.files().get_media(fileId=file_id)
        output_path = os.path.join(ruta_video, file_name)
        fh = io.FileIO(output_path, 'wb')
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        last_progress = 0

        while not done:
            status, done = downloader.next_chunk()
            if status:
                delta = int(status.resumable_progress) - last_progress
                if delta > 0:
                    progress.update(delta)
                    last_progress = int(status.resumable_progress)

        progress.button_text = "Descargado"
        progress.complete(os.path.splitext(file_name)[0])

    except Exception as e:
        print(f"error al descargar el video: {e}")

# función principal para el botón
def descargar_video(b):
    clear_output()
    file_link = link_input.value.strip()
    if not file_link:
        print("no se ingresó ningún enlace")
        return
    try:
        file_id = extract_file_id_from_url(file_link)
        download_from_drive(file_id)
    except Exception as e:
        print(f"error: {e}")

# vincular evento click del botón
download_btn.on_click(descargar_video)

#Fin

In [ ]:
# Extracción de Frames

#@markdown ### **Extracción de frames**
#@markdown este script permite extraer frames de un único video mp4 utilizando ffmpeg con aceleración por `gpu`
#@markdown el video se toma automáticamente desde la carpeta `FFmpeg` y los frames se guardan en drive dentro de `Capture`.

# importaciones
import os, subprocess
from IPython.display import clear_output

# parámetros de extracción
frames_video = 1  # @param {type: "slider", min: 0.2, max: 2, step: 0.2}

#@markdown los frames se dividen automáticamente en `4 subcarpetas` y se muestra una barra de progreso animada durante toda la extracción y división.

# rutas
ruta_drive = "drive/MyDrive/"
ruta_video = "/content/FFmpeg"
ruta_frame = os.path.join(ruta_drive, "Capture")
os.makedirs(ruta_frame, exist_ok=True)

# buscar videos mp4 en la carpeta
videos = [f for f in os.listdir(ruta_video) if f.endswith(".mp4")]

if not videos:
    print("no se encontraron videos mp4")
elif len(videos) > 1:
    print("debe existir solo un archivo mp4 en /content/FFmpeg")
else:
    video_path = os.path.join(ruta_video, videos[0])
    dividir_fijo = 4

    load_style()
    clear_output(wait=True)

    # crear carpeta para los frames
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    subcarpeta_frames = os.path.join(ruta_frame, video_name)
    os.makedirs(subcarpeta_frames, exist_ok=True)

    # obtener duración del video
    ffprobe_command = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        video_path]
    duration = float(subprocess.run(ffprobe_command, stdout=subprocess.PIPE, text=True).stdout.strip())

    # barra de progreso extracción
    progress = Animated_progress(total=int(duration), title="Extrayendo frames", button_text="")

    # comando ffmpeg para extraer frames
    ffmpeg_command = [
        'ffmpeg',
        '-hwaccel', 'cuda',
        '-threads', '0',
        '-i', video_path,
        '-vf', f'fps={frames_video}',
        '-q:v', '1',
        '-preset', 'fast',
        '-f', 'image2',
        os.path.join(subcarpeta_frames, "frame %04d.jpg")]
    process = subprocess.Popen(ffmpeg_command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

    while True:
        output = process.stderr.readline()
        if output == '' and process.poll() is not None:
            break
        if "time=" in output:
            time_str = output.split("time=")[1].split(" ")[0]
            h, m, s = map(float, time_str.split(':'))
            current_time = int(h * 3600 + m * 60 + s)
            delta = current_time - progress.current
            if delta > 0:
                progress.update(delta)

    process.wait()
    clear_output(wait=True)

    progress.button_text = "extraído"
    progress.complete(video_name)

    # división de frames en 4 subcarpetas
    frames = sorted(f for f in os.listdir(subcarpeta_frames) if f.endswith('.jpg'))
    total_frames = len(frames)
    frames_por_parte = total_frames // dividir_fijo

    progress_div = Animated_progress(total=total_frames, title="Dividiendo frames", button_text="")

    for i in range(dividir_fijo):
        sub = os.path.join(subcarpeta_frames, f"frames {i+1}")
        os.makedirs(sub, exist_ok=True)
        inicio = i * frames_por_parte
        fin = inicio + frames_por_parte if i < dividir_fijo - 1 else total_frames
        for frame in frames[inicio:fin]:
            os.rename(os.path.join(subcarpeta_frames, frame), os.path.join(sub, frame))
            progress_div.update(1)

    progress_div.button_text = "Finalizado"
    progress_div.complete(video_name)

### **Mejorar Imágenes 🗺️**

<img src="https://i.pinimg.com/originals/76/9a/f2/769af25c9fbe718c28275465c0c3e34d.gif" width="150%" height="230‰" style="margin-bottom: 0;">

Mejorar la calidad, ajustar el tamaño y reducir la resolución de imágenes. Utiliza modelos Real-ESRGAN para mejorar la calidad y OpenCV para ajustar y reducir el tamaño de las imágenes.

<small>
  <a style="text-decoration:none;">
    <img src="https://img.shields.io/badge/4.5-353535?style=for-the-badge&logo=Android&logoColor=%23fafafa&label=Versión&labelColor=292929" alt="Versión 1.0" width="100">
  </a>
</small>
<br>
<small>
  <a style="text-decoration:none;">
    <img src="https://img.shields.io/badge/3-353535?style=for-the-badge&logo=Github&label=Paso&labelColor=292929" alt="Google Drive Management" width="75">
  </a>
</small>

---

</details>  

<details>
  <summary><font color=gray>Últimos cambios</font></summary>

```diff
+ Implementación de una interfaz interactiva para seleccionar opciones.  
+ Integración de modelos avanzados como Real-ESRGAN y OpenCV.  
+ Opción para reemplazar o guardar imágenes procesadas en carpetas específicas.  
+ Barra de progreso personalizada para el seguimiento del procesamiento.  
```

</details>  

<details>
  <summary><font color=gray>Cambios antiguos</font></summary>

```diff
+ Conexión automatizada a Google Drive desde Google Colab.  
+ Creación de rutas dinámicas para la organización de archivos.  
+ Soporte para formatos de imagen como JPG, PNG y JPEG.  
```

---

In [ ]:
# Renombrar y convertir imágenes del dataset

#@markdown ### **Renombrar y convertir**
#@markdown este script permite renombrar las imágenes del dataset usando un nombre base personalizado.
#@markdown el formato final del nombre será: `nom 01, nom 02..`

#@markdown si no se define un nombre base, se utiliza el nombre del `proyecto`
#@markdown permite convertir las imágenes al formato `png` durante el proceso.
#@markdown incluye una barra de progreso personalizada para visualizar el avance.

import os
from PIL import Image
from IPython.display import clear_output, display

# importar widgets visuales
from ColabUI.Animated_box import pink_button_download
from ColabUI.Animated_progress import Animated_progress

# formato final
formato_base = "png"

# rutas personales
ruta_drive = "/content/drive/MyDrive/"
ruta_proy  = os.path.join(ruta_drive, "Loras")
ruta_sour  = os.path.join(ruta_proy, ruta_name, "dataset")

# crear input y botón rosa
rename_btn, nombre_input = pink_button_download(
    title="Renombrar imágenes",
    btn_text="Renombrar",
    input_placeholder="Nuevo nombre"
)

def renombrar_imagenes(b):
    clear_output(wait=True)

    nombre_base = nombre_input.value.strip()

    # usar ruta_name si no se define nombre
    if not nombre_base:
        nombre_base = ruta_name

    # normalizar espacios y pasar a minúsculas
    nombre_base = " ".join(nombre_base.split()).lower()

    # extensiones aceptadas
    ext_validas = ('.png', '.jpg', '.jpeg', '.webp')

    # imágenes
    imagenes = sorted([
        f for f in os.listdir(ruta_sour)
        if f.lower().endswith(ext_validas)
    ])

    # barra de progreso personalizada
    progress_bar = Animated_progress(
        total=len(imagenes),
        title="Convirtiendo",
        button_text=""
    )

    contador = 1
    for file in imagenes:
        img_path = os.path.join(ruta_sour, file)
        img = Image.open(img_path).convert("RGB")

        nuevo_nombre = f"{nombre_base} {contador:02d}.{formato_base}"
        nuevo_path = os.path.join(ruta_sour, nuevo_nombre)

        img.save(nuevo_path, quality=100)

        # eliminar archivo original si el nombre o formato cambia
        if nuevo_path != img_path:
            os.remove(img_path)

        contador += 1
        progress_bar.update(1)

    # finalizar barra
    progress_bar.button_text = f"{len(imagenes)} imágenes"
    progress_bar.complete("Proceso completado")

# vincular evento
rename_btn.on_click(renombrar_imagenes)

# Fin

In [ ]:
# Mejorar imágenes

# bloque 1 instalación del programa
# =================================

# importaciones
import os, shutil, sys, re, random, numpy as np, torch
from ColabUI.Widgets.loading_bars import loading_bars
loading_bars("instalando el programa", padding_left=8)

# bandera de instalación
setup_flag = "/content/Real-ESRGAN/setup_done.txt"
if not os.path.exists(setup_flag):

    # instalación y configuración
    os.system("nvidia-smi")
    os.system("git clone https://github.com/xinntao/Real-ESRGAN")
    os.chdir("/content/Real-ESRGAN")
    os.system("pip install basicsr facexlib gfpgan")
    os.system("pip install -r requirements.txt")
    os.system("python setup.py develop")

    import PIL.Image, IPython.display
    from tqdm import tqdm
    from google.colab import files
    from torch.nn import functional as F

    new_model_path = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth'
    filename = '/content/Real-ESRGAN/inference_realesrgan.py'

    # reemplaza la ruta del modelo en el script de inferencia
    with open(filename) as f: script = f.read()
    script = re.sub(r"(model_path\s*=\s*[\"\']).*?([\"\'])", rf"\1{new_model_path}\2", script)
    with open(filename,'w') as f: f.write(script)

    os.system('sed -i "s/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/" /usr/local/lib/python3.10/dist-packages/basicsr/data/degradations.py')

    # degradations personalizado
    degradations_code = '''import cv2, math, random, torch, numpy as np
from torch.nn import functional as F

def random_add_gaussian_noise_pt(img, sigma_range=(0,1.0), gray_prob=0, noise_gray_prob=0, clip=True, rounds=False):
    sigma = random.uniform(*sigma_range)
    if random.random()<gray_prob: img=rgb_to_grayscale(img)
    noise = (torch.randn(*img.shape[1:],device=img.device).unsqueeze(0).repeat(img.shape[0],1,1)
             if random.random()<noise_gray_prob else torch.randn_like(img))*sigma
    out = img+noise
    if clip and rounds: return torch.clamp((out*255).round(),0,255)/255
    if clip: return torch.clamp(out,0,1)
    if rounds: return (out*255).round()/255
    return out

def random_add_poisson_noise_pt(img, scale_range=(0,1.0), gray_prob=0, clip=True, rounds=False):
    scale=random.uniform(*scale_range)
    if random.random()<gray_prob: img=rgb_to_grayscale(img)
    out=img+(torch.poisson(img*scale)/scale-img)
    if clip and rounds: return torch.clamp((out*255).round(),0,255)/255
    if clip: return torch.clamp(out,0,1)
    if rounds: return (out*255).round()/255
    return out

def rgb_to_grayscale(img):
    if img.shape[0]!=3: raise ValueError('input image must have 3 channels')
    w=torch.tensor([0.2989,0.5870,0.1140],device=img.device).view(-1,1,1)
    return torch.sum(img*w,dim=0,keepdim=True)

def circular_lowpass_kernel(cutoff,kernel_size,pad_to=0):
    pad_to=pad_to or kernel_size
    assert pad_to>=kernel_size
    def sinc(x): return torch.tensor(1.) if x==0 else torch.sin(x*math.pi*x)/(x*math.pi*x)
    g=torch.linspace(-(kernel_size-1)/2,(kernel_size-1)/2,kernel_size)
    x,y=torch.meshgrid(g,g)
    k=sinc(torch.sqrt(x**2+y**2)*cutoff); k/=k.sum()
    if pad_to>kernel_size: k=F.pad(k,[(pad_to-kernel_size)//2]*4)
    return k

def random_mixed_kernels(kernel_list,kernel_prob,kernel_size=21,blur_sigma=0.1,blur_sigma_min=0.1,blur_sigma_max=10.0,pad_to=0):
    pad_to=pad_to or kernel_size
    t=np.random.choice(kernel_list,p=kernel_prob)
    if t=='iso': return _iso(kernel_size,np.random.uniform(blur_sigma_min,blur_sigma_max),pad_to)
    if t=='aniso': return _aniso(kernel_size,np.random.uniform(blur_sigma_min,blur_sigma_max),
                                 np.random.uniform(blur_sigma_min,blur_sigma_max),
                                 np.random.uniform(-np.pi,np.pi),pad_to)
    return circular_lowpass_kernel(blur_sigma,kernel_size,pad_to)

def _iso(k,s,p):
    g=torch.linspace(-(k-1)/2,(k-1)/2,k)
    x,y=torch.meshgrid(g,g)
    k=torch.exp(-(x**2+y**2)/(2*s**2)); k/=k.sum()
    return F.pad(k,[(p-k.shape[0])//2]*4) if p>k.shape[0] else k

def _aniso(k,sx,sy,r,p):
    g=torch.linspace(-(k-1)/2,(k-1)/2,k)
    x,y=torch.meshgrid(g,g)
    c,s=torch.cos(torch.tensor(r)),torch.sin(torch.tensor(r))
    xr,yr=c*x-s*y,s*x+c*y
    k=torch.exp(-(xr**2/(2*sx**2)+yr**2/(2*sy**2))); k/=k.sum()
    return F.pad(k,[(p-k.shape[0])//2]*4) if p>k.shape[0] else k
'''
    pyver=f"python{sys.version_info.major}.{sys.version_info.minor}"
    path=f'/usr/local/lib/{pyver}/dist-packages/basicsr/data/degradations.py'
    os.system(f"cp -n {path} {path}.backup")
    with open(path,'w') as f: f.write(degradations_code)

    # crea archivo de bandera
    with open(setup_flag, 'w') as f: f.write("setup done")

# bloque 2 iniciar proceso de imágenes
# ====================================
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import clear_output

# directorio base
os.chdir("/content")

# carpeta dinámica de entrada / salida (directa)
ruta_drive = "/content/drive/MyDrive/"
ruta_proy  = os.path.join(ruta_drive, "Loras")
ruta_sour  = os.path.join(ruta_proy, ruta_name, "dataset")

#@markdown ### **Mejorar calidad de imágenes**
#@markdown este script permite mejorar la calidad de imágenes utilizando el modelo real-esrgan.
#@markdown las imágenes procesadas se almacenan directamente en el dataset original, sobrescribiendo los archivos.

# parámetros del modelo
model_name = "realesr-animevideov3"
escalado = 2  #@param {type:"slider", min:1, max:4, step:1}
face_enhance = "No"

#@markdown el usuario también puede ajustar el nivel de escalado para aumentar la resolución de las imágenes.
# lista de imágenes a procesar
imagenes = [
    f for f in os.listdir(ruta_sour)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# crear barra de progreso
clear_output()
progress_bar = Animated_progress(
    total=len(imagenes),
    title="Mejorando",
    button_text="")

# función para procesar una imagen
def procesar_imagen(file):
    input_path = os.path.join(ruta_sour, file)
    command = (
        f"python Real-ESRGAN/inference_realesrgan.py "
        f"-n {model_name} "
        f"-i '{input_path}' "
        f"--outscale {escalado} "
        f"--output '{ruta_sour}'")

    if face_enhance == "Yes":
        command += " --face_enhance"
    os.system(command)

    # borrar imagen original
    if os.path.exists(input_path):
        os.remove(input_path)
    progress_bar.update(1)

# procesamiento en paralelo
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(procesar_imagen, file) for file in imagenes]
    for future in as_completed(futures):
        pass

# finalizar barra con botón mostrando total de imágenes
progress_bar.button_text = f"{len(imagenes)} imágenes"
progress_bar.complete("Mejora completada")

# Fin

In [ ]:
00# Redimensionar imágenes

#@markdown ### **Redimensionar imágenes**
#@markdown Este script ajusta las imágenes para que el lado más corto siempre sea de **1000 píxeles**, manteniendo la proporción original de la imagen.

import os
from PIL import Image
from IPython.display import clear_output

# importar barra de progreso personalizada
from ColabUI.Animated_progress import Animated_progress

# rutas personales
ruta_drive = "/content/drive/MyDrive/"
ruta_proy  = os.path.join(ruta_drive, "Loras")
ruta_sour  = os.path.join(ruta_proy, ruta_name, "dataset")

# parámetros
# lado más corto
tamaño = 1000  #@param {type:"slider", min:900, max:1200, step:100}
LADO_CORTO = tamaño
FORMATOS = list(range(900, 3001, 100))

#@markdown este script redimensiona imágenes forzando que el lado más pequeño sea **1000 px**
#@markdown el lado largo se **estira y redondea** al formato más cercano permitido

# imágenes válidas
imagenes = [
    f for f in os.listdir(ruta_sour)
    if f.lower().endswith(('.png', '.jpg'))]

# seleccionar formato más cercano
def formato_mas_cercano(valor):
    return min(FORMATOS, key=lambda x: abs(x - valor))

# barra de progreso personalizada
clear_output()
progress_bar = Animated_progress(
    total=len(imagenes),
    title="Redimensionando",
    button_text="")

# procesar imágenes (secuencial)
for file in imagenes:
    img_path = os.path.join(ruta_sour, file)
    img = Image.open(img_path).convert("RGB")
    w, h = img.size

    if w <= h:
        lado_corto = w
        lado_largo = h
        vertical = True
    else:
        lado_corto = h
        lado_largo = w
        vertical = False

    escala = LADO_CORTO / lado_corto
    largo_estimado = int(lado_largo * escala)
    largo_final = formato_mas_cercano(largo_estimado)

    if vertical:
        size_final = (LADO_CORTO, largo_final)
    else:
        size_final = (largo_final, LADO_CORTO)

    img = img.resize(size_final, Image.BICUBIC)
    img.save(img_path, quality=95)
    progress_bar.update(1)

# finalizar barra
progress_bar.button_text = f"{len(imagenes)} imágenes"
progress_bar.complete("Proceso completado")

# Fin

### **Etiquetar y Editar 🏷️**

<img src="https://i.pinimg.com/originals/3c/d2/2a/3cd22aba559021ceb78189475ad918f4.gif" width="150%" height="230‰" style="margin-bottom: 0;">

Etiquetar automáticamente imágenes utilizando el modelo WaifuDiffusion Tagger y edita las etiquetas generadas. Las etiquetas se guardan en archivos de texto a cada imagen.

<small>
  <a style="text-decoration:none;">
    <img src="https://img.shields.io/badge/4.0-353535?style=for-the-badge&logo=Android&logoColor=%23fafafa&label=Versión&labelColor=292929" alt="Versión 1.0" width="100">
  </a>
</small>
<br>
<small>
  <a style="text-decoration:none;">
    <img src="https://img.shields.io/badge/4-353535?style=for-the-badge&logo=Github&label=Paso&labelColor=292929" alt="Google Drive Management" width="75">
  </a>
</small>

---

</details>  

<details>
  <summary><font color=gray>Últimos cambios</font></summary>

```diff
+ Implementación de una interfaz interactiva para seleccionar y editar etiquetas.  
+ Integración del modelo **WaifuDiffusion Tagger** para etiquetado automático.  
+ Opción para añadir nuevas etiquetas personalizadas.  
+ Visualización de las etiquetas más comunes con badges interactivos.  
```

</details>  

<details>
  <summary><font color=gray>Cambios antiguos</font></summary>

```diff
+ Conexión automatizada a Google Drive desde Google Colab.  
+ Creación de archivos de texto para almacenar etiquetas.  
+ Soporte para formatos de imagen como JPG, PNG y JPEG.  
```

In [ ]:
# Etiquetado de imágenes

#@markdown ### **Etiquetar imágenes**
#@markdown este script analiza automáticamente las imágenes del dataset usando el modelo `PixAI Tagger v0.9` optimizado para waifudiffusion.
#@markdown las etiquetas cubren personajes, atributos visuales, vestimenta, expresiones, objetos y otros detalles útiles para entrenamiento.

umbral = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
umbral_personaje = 1
umbral_general   = 1 - umbral

# configuración inicial
#@markdown incluye barra de progreso animada, compatibilidad con gpu y genera archivos de etiquetas listas para usar en lora.
#@markdown puedes ajustar el umbral de detección usando el control deslizante a continuación.

ruta_drive = "drive/MyDrive/"
ruta_proy = os.path.join(ruta_drive, "Loras")
ruta_data = os.path.join(ruta_proy, ruta_name, "dataset")

# instalar dependencias
!pip install -q huggingface-hub torch torchvision pillow timm requests

# importaciones
import os, time, json, glob, requests, timm, torch
import torchvision.transforms as transforms
from pathlib import Path
from PIL import Image
from typing import Dict, Any
from IPython.display import clear_output, display, HTML

# descarga de archivos
MODEL_DIR = "./assets"
os.makedirs(MODEL_DIR, exist_ok=True)

FILE_URLS = {
    "model_v0.9.pth": "https://huggingface.co/SFcrypt/colab/resolve/main/model_v0.9.pth?download=true",
    "tags_v0.9_13k.json": "https://huggingface.co/SFcrypt/colab/resolve/main/tags_v0.9_13k.json?download=true",
    "char_ip_map.json": "https://huggingface.co/SFcrypt/colab/resolve/main/char_ip_map.json?download=true",
}

def download_file(url: str, dest_path: Path):
    print(f"Descargando {dest_path.name}...")
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"OK {dest_path.name}")
        return True
    except Exception as e:
        print(f"Error descargando {dest_path.name}: {e}")
        return False

def ensure_assets():
    target = Path(MODEL_DIR)
    target.mkdir(parents=True, exist_ok=True)
    all_ok = True
    for filename, url in FILE_URLS.items():
        dest_path = target / filename
        if dest_path.exists():
            print(f"OK {filename} existe")
            continue
        if download_file(url, dest_path):
            print(f"OK {filename} descargado")
        else:
            all_ok = False
            print(f"Fallo {filename}")
    return all_ok

ensure_assets()

# modelo
class TaggingHead(torch.nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.head = torch.nn.Sequential(torch.nn.Linear(input_dim, num_classes))
    def forward(self, x):
        return torch.nn.functional.sigmoid(self.head(x))

def get_encoder():
    encoder = timm.create_model("hf_hub:SmilingWolf/wd-eva02-large-tagger-v3", pretrained=False)
    encoder.reset_classifier(0)
    return encoder

def get_model():
    model = torch.nn.Sequential(get_encoder(), TaggingHead(1024, 13461))
    return model

def load_model(weights_file, device):
    model = get_model()
    model.load_state_dict(torch.load(weights_file, map_location=device, weights_only=True))
    model.to(device)
    model.eval()
    return model

# utilidades
def get_tags(tags_file: Path):
    with tags_file.open("r", encoding="utf-8") as f:
        tag_info = json.load(f)
    tag_map = tag_info["tag_map"]
    gen_tag_count = tag_info["tag_split"]["gen_tag_count"]
    character_tag_count = tag_info["tag_split"]["character_tag_count"]
    return tag_map, gen_tag_count, character_tag_count

def get_character_ip_mapping(mapping_file: Path):
    with mapping_file.open("r", encoding="utf-8") as f:
        return json.load(f)

def pil_to_rgb(image):
    if image.mode == "RGBA" or image.mode == "P":
        image = image.convert("RGBA")
        image.load()
        background = Image.new("RGB", image.size, (255, 255, 255))
        background.paste(image, mask=image.split()[3])
        return background
    return image.convert("RGB")

# tagger
class PixAITagger:
    def __init__(self, model_dir: str):
        repo_path = Path(model_dir)
        weights_file = repo_path / "model_v0.9.pth"
        tags_file = repo_path / "tags_v0.9_13k.json"
        mapping_file = repo_path / "char_ip_map.json"

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Dispositivo: {self.device}")

        print("Cargando modelo...")
        self.model = load_model(str(weights_file), self.device)

        self.transform = transforms.Compose([
            transforms.Resize((448, 448)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

        tag_map, self.gen_tag_count, self.character_tag_count = get_tags(tags_file)
        self.index_to_tag_map = {v: k for k, v in tag_map.items()}
        self.character_ip_mapping = get_character_ip_mapping(mapping_file)
        self.default_character_threshold = 0.85
        print(f"Cargadas {len(tag_map)} etiquetas")

    def process_image(self, image_path: str, general_threshold: float = 0.30) -> Dict[str, Any]:
        try:
            image = pil_to_rgb(Image.open(image_path))
        except Exception as e:
            print(f"Error cargando {image_path}: {e}")
            return None

        with torch.inference_mode():
            image_tensor = self.transform(image).unsqueeze(0).to(self.device)
            probs = self.model(image_tensor)[0]

        # etiquetas generales
        general_mask = probs[:self.gen_tag_count] > general_threshold
        general_indices = general_mask.nonzero(as_tuple=True)[0]
        general_tags = sorted([self.index_to_tag_map[int(idx.item())] for idx in general_indices])

        # etiquetas de personajes
        character_mask = probs[self.gen_tag_count:] > self.default_character_threshold
        character_indices = character_mask.nonzero(as_tuple=True)[0] + self.gen_tag_count
        character_tags = sorted([self.index_to_tag_map[int(idx.item())] for idx in character_indices])

        # etiquetas ip
        ip_tags = sorted(set([tag for tag in character_tags if tag in self.character_ip_mapping]))

        return {"feature": general_tags, "character": character_tags, "ip": ip_tags}

# importar barra de progreso
try:
    from ColabUI.Animated_progress import Animated_progress
except ImportError:
    # si no existe ColabUI, definir clase simplificada
    class Animated_progress:
        def __init__(self, total, title="Título", button_text=""):
            self.total = total
            self.current = 0
            self.button_text = button_text
            self.title = title
            self._display()

        def _display(self):
            clear_output(wait=True)
            percent = int((self.current / self.total) * 100) if self.total > 0 else 0
            bar_length = 30
            filled = int(bar_length * self.current / self.total) if self.total > 0 else 0
            bar = '█' * filled + '░' * (bar_length - filled)
            print(f"{self.title}: [{bar}] {percent}% ({self.current}/{self.total})")
            print(self.button_text)

        def update(self, step=1, text=None):
            self.current = min(self.current + step, self.total)
            if text:
                self.button_text = text
            self._display()

        def complete(self, message="Completado"):
            self.current = self.total
            self._display()
            print(f"✅ {message}")

# procesamiento por lotes
def process_dataset(tagger: PixAITagger, dataset_path: str, general_threshold: float = 0.30):
    if not os.path.exists(dataset_path):
        print(f"La ruta no existe: {dataset_path}")
        return

    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp', '*.bmp']
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(dataset_path, ext)))
        image_files.extend(glob.glob(os.path.join(dataset_path, ext.upper())))
    image_files = list(set(image_files))

    if not image_files:
        print(f"No se encontraron imagenes en: {dataset_path}")
        return

    clear_output()
    total_images = len(image_files)
    progress_bar = Animated_progress(total=total_images, title="Etiquetando", button_text="")

    for img_path in image_files:
        try:
            result = tagger.process_image(img_path, general_threshold)
            if result is None:
                progress_bar.update()
                continue

            txt_path = os.path.splitext(img_path)[0] + ".txt"
            tags = result.get("feature", [])
            formatted_tags = ", ".join(tag.replace("_", " ") for tag in tags)

            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(formatted_tags)

            progress_bar.update()

        except Exception as e:
            progress_bar.update()
            continue

    progress_bar.button_text = f"{total_images} imágenes"
    progress_bar.complete("Proyecto listo")

# ejecucion
def main():
    if not os.path.exists(ruta_data):
        print(f"Creando directorio: {ruta_data}")
        os.makedirs(ruta_data, exist_ok=True)
        print("Coloca tus imagenes en esta carpeta y vuelve a ejecutar.")
        return

    tagger = PixAITagger(MODEL_DIR)
    process_dataset(tagger, ruta_data, umbral_general)

if __name__ == "__main__":
    main()

In [ ]:
#@markdown ### **Editar etiquetas**
#@markdown este script permite visualizar imagenes del dataset y editar etiquetas asociadas a cada archivo.
#@markdown facilita la navegacion entre imagenes, el guardado automatico de etiquetas y la gestion del progreso.

#@markdown incluye herramientas para agregar y eliminar etiquetas individuales o globales de forma sencilla.
#@markdown integra enlaces a la wiki de danbooru para consultar informacion de cada etiqueta facilmente.

!pip install --upgrade gradio pandas==2.2.2 -q
import os, json, gradio as gr, pandas as pd, re, threading, time, sys, io
from IPython.display import HTML, display
import ipywidgets as widgets

# Configuración
ruta_drive = "drive/MyDrive/"
ruta_proy = os.path.join(ruta_drive, "Loras")
ruta_data = os.path.join(ruta_proy, ruta_name, "dataset")

image_list = sorted([os.path.join(ruta_data, f) for f in os.listdir(ruta_data) if f.lower().endswith((".png", ".jpg", ".jpeg"))])

ruta_widget = "/content/ColabUI"
os.makedirs(ruta_widget, exist_ok=True)
state_file = os.path.join(ruta_widget, "INDEX.md")

def save_index(idx):
    with open(state_file, "w") as f:
        json.dump({"index": idx}, f)

def load_index():
    if os.path.exists(state_file):
        return json.load(open(state_file, "r")).get("index", 0)
    return 0

# funciónes Principales
def generar_link(tag):
    if not tag.strip():
        return ""
    clean_tag = tag.replace(",", " ").strip()
    url = f"https://danbooru.donmai.us/wiki_pages/{clean_tag.replace(' ', '_')}"
    return f'<a href="{url}" target="_blank" class="tag-link">{tag.strip()}</a>'

def generar_links_tags(tags_str):
    if not tags_str or not tags_str.strip():
        return ""
    tags = [t.strip() for t in tags_str.split(",") if t.strip()]
    return ", ".join([generar_link(tag) for tag in tags])

def load_image(idx):
    if not image_list:
        return None, idx, "", "", "⚠️ No hay imágenes"
    idx = idx % len(image_list)
    img = image_list[idx]
    txt = os.path.splitext(img)[0] + ".txt"
    tags = open(txt, "r", encoding="utf-8").read() if os.path.exists(txt) else ""
    save_index(idx)
    name = os.path.splitext(os.path.basename(img))[0]
    return img, idx, tags, generar_links_tags(tags), f"📜 Imagen {name}"

def save_current(idx, tags):
    if not image_list:
        return
    txt = os.path.splitext(image_list[idx % len(image_list)])[0] + ".txt"
    with open(txt, "w", encoding="utf-8") as f:
        f.write(tags or "")

def next_image(idx, tags):
    save_current(idx, tags)
    return load_image(idx + 1)

def prev_image(idx, tags):
    save_current(idx, tags)
    return load_image(idx - 1)

def agregar_tag_dataset(tag):
    if not tag or not tag.strip():
        return "⚠️ Etiqueta vacía"

    if not image_list:
        return "⚠️ No hay imágenes"
    tag_clean, count = tag.strip(), 0
    for img in image_list:
        txt = os.path.splitext(img)[0] + ".txt"
        contenido = open(txt, "r", encoding="utf-8").read().strip() if os.path.exists(txt) else ""
        tags_list = [t.strip() for t in contenido.split(",") if t.strip()]

        if tag_clean not in tags_list:
            nuevo = f"{tag_clean}, {contenido}" if contenido else tag_clean
            with open(txt, "w", encoding="utf-8") as f:
                f.write(nuevo)
            count += 1
    return f"✨ Agregada '{tag_clean}' a {count} imágenes" if count > 0 else f"ℹ️ '{tag_clean}' ya existe"

def eliminar_tag_dataset(tag):
    if not tag or not tag.strip():
        return "⚠️ Etiqueta vacía"
    if not image_list:
        return "⚠️ No hay imágenes"
    tag_clean, count = tag.strip(), 0

    for img in image_list:
        txt = os.path.splitext(img)[0] + ".txt"
        if not os.path.exists(txt):
            continue
        contenido = open(txt, "r", encoding="utf-8").read().strip()
        tags_list = [t.strip() for t in contenido.split(",") if t.strip()]
        if tag_clean in tags_list:
            tags_list = [t for t in tags_list if t != tag_clean]
            with open(txt, "w", encoding="utf-8") as f:
                f.write(", ".join(tags_list))
            count += 1
    return f"💀 Eliminada '{tag_clean}' de {count} imágenes" if count > 0 else f"ℹ️ '{tag_clean}' no encontrada"

def reemplazar_tag_dataset(tag_original, tag_nueva):
    if not tag_original or not tag_original.strip():
        return "⚠️ Etiqueta original vacía"
    if not image_list:
        return "⚠️ No hay imágenes"

    tag_orig = tag_original.strip()

    # Si no hay etiqueta nueva, eliminar la etiqueta original
    if not tag_nueva or not tag_nueva.strip():
        return eliminar_tag_dataset(tag_orig)

    tag_new = tag_nueva.strip()
    count = 0

    for img in image_list:
        txt = os.path.splitext(img)[0] + ".txt"
        if not os.path.exists(txt):
            continue
        contenido = open(txt, "r", encoding="utf-8").read().strip()
        tags_list = [t.strip() for t in contenido.split(",") if t.strip()]

        if tag_orig in tags_list:
            # Reemplazar la etiqueta original por la nueva
            tags_list = [tag_new if t == tag_orig else t for t in tags_list]
            # Eliminar duplicados de la etiqueta nueva (manteniendo solo una ocurrencia)
            tags_list = list(dict.fromkeys(tags_list))
            with open(txt, "w", encoding="utf-8") as f:
                f.write(", ".join(tags_list))
            count += 1

    return f"🔄 Reemplazada '{tag_orig}' → '{tag_new}' en {count} imágenes" if count > 0 else f"ℹ️ '{tag_orig}' no encontrada"

# wrappers Gradio
def next_wrapper(idx, tags):
    save_current(idx, tags)
    return load_image(idx + 1)

def prev_wrapper(idx, tags):
    save_current(idx, tags)
    return load_image(idx - 1)

def agregar_wrapper(tag, idx):
    if not tag or not tag.strip():
        img, new_idx, new_tags, links, msg = load_image(idx)
        return "⚠️ Ingresa una etiqueta", img, new_idx, new_tags, links, msg
    res = agregar_tag_dataset(tag)
    img, new_idx, new_tags, links, msg = load_image(idx)
    return res, img, new_idx, new_tags, links, msg

def reemplazar_wrapper(tag_orig, tag_new, idx):
    if not tag_orig or not tag_orig.strip():
        img, new_idx, new_tags, links, msg = load_image(idx)
        return "⚠️ Ingresa una etiqueta", img, new_idx, new_tags, links, msg
    res = reemplazar_tag_dataset(tag_orig, tag_new)
    img, new_idx, new_tags, links, msg = load_image(idx)
    return res, img, new_idx, new_tags, links, msg

def actualizar_tags(tags):
    return generar_links_tags(tags)

# --- CSS ---
custom_css = """
/* quitar color azul al escribir */
textarea, textarea:focus {
    background-color: var(--input-background-fill) !important;
    color: var(--body-text-color) !important;}

/* AUMENTAR TAMAÑO DEL CURSOR EN EL CAMPO DE ETIQUETAS */
#sorted_out textarea {
    caret-color: #ff6b9d !important;
    font-size: 18px !important;
    line-height: 1.4 !important;
    padding: 13px !important;}

/* QUITAR EL MARCO ROSA AL HACER FOCUS */
#sorted_out textarea:focus {
    box-shadow: none !important;
    border-color: var(--input-border-color) !important;}

/* quitar selección azul */
textarea::selection {
    background: #555 !important;
    color: #fff !important;}

textarea::-moz-selection {
    background: #555 !important;
    color: #fff !important;}

/* Estilo para los enlaces de tags */
.tag-link {
    color: #ff6b9d !important;
    text-decoration: none !important;
    font-weight: 500 !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;}

.tag-link:hover {
    color: #ff4080 !important;
    text-decoration: underline !important;
    background: rgba(255, 107, 157, 0.1) !important;
    padding: 2px 4px !important;
    border-radius: 4px !important;}

.tag-link:visited {
    color: #808080 !important;}

button {
    transition: all 0.2s ease !important;}

button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 4px 12px rgba(0,0,0,0.2) !important;}

#estado_textbox textarea {
    color: #ffffff !important;
    font-weight: 600 !important;}
"""
# Interfaz Gradio
with gr.Blocks() as demo:
    gr.Markdown("# 🌸 WaifuDiffusion")
    with gr.Row():
        with gr.Column(variant="panel"):
            image_in = gr.Image(type="filepath", label="imagen", height=400)

            with gr.Row():
                prev_btn = gr.Button("Atrás", variant="secondary")
                next_btn = gr.Button("Siguiente", variant="secondary")

        with gr.Column(variant="panel"):
            with gr.Accordion("Ver etiquetas", open=True):
                wiki_tags_box = gr.HTML("")
            sorted_out = gr.Textbox(label="Editar etiquetas", lines=4, elem_id="sorted_out")

            # Acordeón para reemplazar/agregar etiquetas
            with gr.Accordion("Reemplazar Etiquetas", open=False):
                with gr.Row():
                    tag_original_input = gr.Textbox(label="Etiqueta original", lines=1, placeholder="Escribe una etiqueta", scale=1)
                    tag_nueva_input = gr.Textbox(label="Etiqueta nueva", lines=1, placeholder="Dejar vacío para eliminar", scale=1)
                with gr.Row():
                    agregar_btn = gr.Button("Agregar", variant="secondary", scale=1)
                    reemplazar_btn = gr.Button("Reemplazar", variant="secondary", scale=1)

            msg_out = gr.Textbox(label="Estado", interactive=False, elem_id="estado_textbox")

    index_state = gr.State(load_index())

    # Eventos
    next_btn.click(next_wrapper, [index_state, sorted_out], [image_in, index_state, sorted_out, wiki_tags_box, msg_out])
    prev_btn.click(prev_wrapper, [index_state, sorted_out], [image_in, index_state, sorted_out, wiki_tags_box, msg_out])
    sorted_out.change(actualizar_tags, [sorted_out], [wiki_tags_box])

    agregar_btn.click(agregar_wrapper, [tag_original_input, index_state], [msg_out, image_in, index_state, sorted_out, wiki_tags_box, msg_out]
    ).then(lambda: "", [], [tag_original_input])

    reemplazar_btn.click(reemplazar_wrapper, [tag_original_input, tag_nueva_input, index_state], [msg_out, image_in, index_state, sorted_out, wiki_tags_box, msg_out]
    ).then(lambda: "", [], [tag_original_input]).then(lambda: "", [], [tag_nueva_input])

    demo.load(lambda: load_image(load_index()), [], [image_in, index_state, sorted_out, wiki_tags_box, msg_out])

demo.queue()

# Lanzamiento
def launch_and_get_url(demo):
    old_stdout = sys.stdout
    sys.stdout = mystdout = io.StringIO()
    t = threading.Thread(target=lambda: demo.launch(share=True, theme="lone17/kotaemon", css=custom_css, prevent_thread_lock=True))
    t.start()
    url = None
    while url is None:
        time.sleep(1)
        match = re.search(r"https://[a-z0-9]+\.gradio\.live", mystdout.getvalue())
        if match:
            url = match.group(0)
    sys.stdout = old_stdout
    return url

app_url = launch_and_get_url(demo)

# crear botón rosa (se muestra automáticamente)
clear_output()
btn_link, input_box = pink_button_download(
    title="Editor de etiquetas",
    btn_text="🌸 WaifuDiffusion",
    input_placeholder="")

# ocultar input
input_box.layout.display = "none"

def abrir_app(b):
    display(HTML(f"""
    <script>
        window.open("{app_url}", "_blank");
    </script>
    """))

btn_link.on_click(abrir_app)

# Fin